<img src="https://govspace.io/wp-content/uploads/2025/09/GovSpace-web.svg" width="200"># Module M2.10 — Graduated autonomy & human-in-the-loop agent design> **💡 Tip:** Create a folder with your name (e.g. `matt-grasser/`) and copy this notebook into it before editing. The `TEMPLATES` folder syncs from GitHub and is read-only — working in your own folder keeps your work safe.This super-notebook combines three sequential walkthroughs into one continuous experience. By the end you will have:- A working environment with the SDKs verified (00-Setup material)- A tool-using agent that can call functions (01-Basic-Agent material — the agent-loop foundation that everything below builds on)- Two human-in-the-loop patterns wired up: an explicit approval gate and confidence-based routing that auto-executes routine actions while escalating risky ones (03-Graduated-Autonomy material)The whole arc takes about 45 minutes if you read and run each cell. The graduated-autonomy patterns assume familiarity with LangGraph (`StateGraph`, `TypedDict`, conditional edges) — see M2.9 for a full walkthrough of those concepts if you have not already.## What you will work through1. **Setup & Verify Your Environment**2. **Basic Agent with Tool Use**3. **Graduated Autonomy — Human-in-the-Loop Patterns**---

---# 🔹 Section: Setup & Verify Your Environment*(Source: `setup.ipynb`)*

## Step 1: Configure API Keys

This is a **shared server** — all participants can see each other's files and notebook outputs. To keep your API keys safe:

1. Paste your keys in the cell below and run it (this sets them in memory only)
2. **Immediately clear the cell output** after running: click the cell output → right-click → "Clear Outputs"
3. Your keys will stay in memory for this session but won't be visible to others

> **Important:** Do NOT write keys to files on this server. Other participants can see them.

In [ ]:
import os

# ✏️ Replace the placeholder values with your real API keys, then run this cell.
# ⚠️ CLEAR THE OUTPUT immediately after running (right-click output → Clear Outputs).

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-PASTE-YOUR-KEY-HERE"
os.environ["OPENAI_API_KEY"] = "sk-PASTE-YOUR-KEY-HERE"

# Verify keys are loaded (shows first/last few chars only)
for key in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    val = os.environ.get(key, "")
    if val and "PASTE" not in val:
        print(f"✅ {key} loaded ({val[:8]}...{val[-4:]})")
    else:
        print(f"❌ {key} not set — replace the placeholder above and re-run")

In [ ]:
import importlib

packages = [
    ("anthropic", "Anthropic SDK"),
    ("openai", "OpenAI SDK"),
    ("langchain_core", "LangChain Core"),
    ("langchain_anthropic", "LangChain Anthropic"),
    ("langchain_openai", "LangChain OpenAI"),
    ("langgraph", "LangGraph"),
    ("httpx", "httpx"),
    ("dotenv", "python-dotenv"),
]

for module, name in packages:
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, "__version__", "installed")
        print(f"✅ {name}: {version}")
    except ImportError:
        print(f"❌ {name}: NOT INSTALLED")

## Step 3: Hello World — Anthropic (Claude)

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly one sentence."}],
)

print(message.content[0].text)

## Step 4: Hello World — OpenAI (GPT)

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly one sentence."}],
)

print(response.choices[0].message.content)

---

✅ **All set!** If both hello-world calls returned responses, you're ready to proceed:

<div style="background-color:#19b3c2; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Next:</strong> 01-Basic-Agent.ipynb — Build a single agent with tool use</div>
<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Then:</strong> 02-LangGraph-Workflow.ipynb — Multi-step agent workflow with LangGraph</div>
<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Then:</strong> 03-Graduated-Autonomy.ipynb — Human-in-the-loop graduated autonomy</div>

---# 🔹 Section: Basic Agent with Tool Use*(Source: `basic_agent.ipynb`)*

In [ ]:
import os
import anthropic

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()

## Step 1: Define Tools

Tools are JSON schemas that tell the model what functions are available and what arguments they accept.

Let's create two simple tools: a calculator and a weather lookup.

In [ ]:
tools = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use this for any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate, e.g. '(25 * 4) + 10'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Ottawa'"
                }
            },
            "required": ["city"]
        }
    }
]

## Step 2: Implement Tool Handlers

These are the actual Python functions that run when the model calls a tool.

In [ ]:
def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool call to the appropriate handler."""
    if tool_name == "calculate":
        try:
            # Safety note: in production, use a proper math parser, not eval()
            result = eval(tool_input["expression"])
            return str(result)
        except Exception as e:
            return f"Error: {e}"

    elif tool_name == "get_weather":
        # Simulated weather data (in a real app, call a weather API)
        fake_weather = {
            "ottawa": "☀️ 22°C, sunny",
            "toronto": "🌤️ 19°C, partly cloudy",
            "vancouver": "🌧️ 14°C, rain",
        }
        city = tool_input["city"].lower()
        return fake_weather.get(city, f"🌡️ 20°C, weather data not available for {tool_input['city']}")

    return f"Unknown tool: {tool_name}"

## Step 3: The Agent Loop

The core pattern: send a message → if the model wants to call a tool → execute it → feed the result back → repeat until the model responds with text.

This is the **agentic loop** — the model decides what to do next.

In [ ]:
def run_agent(user_message: str, max_turns: int = 5) -> str:
    """Run a simple tool-use agent loop."""
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # If the model just responds with text, we're done
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\n".join(text_blocks)

        # If the model wants to use tools, execute them
        if response.stop_reason == "tool_use":
            # Add the assistant's response (with tool_use blocks) to messages
            messages.append({"role": "assistant", "content": response.content})

            # Process each tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Calling: {block.name}({block.input})")
                    result = handle_tool_call(block.name, block.input)
                    print(f"  📎 Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Feed results back to the model
            messages.append({"role": "user", "content": tool_results})

    return "Agent reached max turns without completing."

## Step 4: Try It Out

Ask the agent questions that require tool use:

In [ ]:
# A question that requires the calculator
print("--- Calculator ---")
result = run_agent("What is 47 * 89 + 123?")
print(f"\n{result}")

In [ ]:
# A question that requires weather lookup
print("--- Weather ---")
result = run_agent("What's the weather like in Ottawa and Vancouver?")
print(f"\n{result}")

In [ ]:
# A question that requires BOTH tools
print("--- Multi-tool ---")
result = run_agent(
    "If it's 22°C in Ottawa, what is that in Fahrenheit? "
    "Also check: what's the actual weather in Toronto?"
)
print(f"\n{result}")

## 💡 Key Takeaways

1. **Tools are schemas** — You describe what's available; the model decides when to use them
2. **The agent loop** — Send → tool call → execute → feed back → repeat
3. **The model orchestrates** — It chooses which tools to call and in what order
4. **Multi-tool calls** — The model can call multiple tools in a single turn

---

<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Next:</strong> 02-LangGraph-Workflow.ipynb — Build a stateful, multi-step workflow where agents can branch, loop, and hand off to each other</div>
<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Then:</strong> 03-Graduated-Autonomy.ipynb — Human-in-the-loop graduated autonomy</div>

---# 🔹 Section: Graduated Autonomy — Human-in-the-Loop Patterns*(Source: `graduated_autonomy.ipynb`)*

In [ ]:
import os

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

from typing import Annotated, TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

model = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=1024)

## Pattern 1: Approval Gate

The simplest human-in-the-loop pattern: the agent proposes an action, and a human approves or rejects it before execution.

```
[Draft] → [Human Review] → [Execute] → END
               ↓
           [Revise] → [Human Review]  (loop)
```

In [ ]:
class EmailState(TypedDict):
    """State for an email drafting workflow."""
    request: str
    messages: Annotated[list, add_messages]
    draft: str
    feedback: str
    approved: bool
    revision_count: int


def draft_email(state: EmailState) -> dict:
    """AI drafts (or revises) an email."""
    revision = state.get("revision_count", 0)

    if revision > 0 and state.get("feedback"):
        prompt = (
            f"Revise this email draft based on feedback.\n\n"
            f"Original request: {state['request']}\n\n"
            f"Current draft:\n{state['draft']}\n\n"
            f"Feedback: {state['feedback']}\n\n"
            f"Write the revised email only, no commentary."
        )
    else:
        prompt = (
            f"Draft a professional email for: {state['request']}\n\n"
            f"Write the email only, no commentary."
        )

    response = model.invoke([HumanMessage(content=prompt)])
    print(f"\n{'📝 Draft' if revision == 0 else f'✏️ Revision {revision}'}:")
    print("-" * 40)
    print(response.content)
    print("-" * 40)

    return {
        "draft": response.content,
        "messages": [response],
        "revision_count": revision + 1,
    }


def human_review(state: EmailState) -> dict:
    """Simulate human review (in production, this would pause for real input)."""
    print("\n🧑 HUMAN REVIEW")
    print("Options: [a]pprove, [r]evise with feedback, [c]ancel")

    # In a notebook, we use input() for interactive review
    choice = input("Your choice: ").strip().lower()

    if choice == "a":
        print("✅ Approved!")
        return {"approved": True, "feedback": ""}
    elif choice.startswith("r"):
        feedback = input("Feedback: ").strip()
        print(f"🔄 Sending back for revision: {feedback}")
        return {"approved": False, "feedback": feedback}
    else:
        print("❌ Cancelled")
        return {"approved": True, "draft": "[CANCELLED]", "feedback": ""}


def execute_send(state: EmailState) -> dict:
    """Execute the approved action (send email)."""
    if state.get("draft") == "[CANCELLED]":
        print("\n🚫 Email cancelled, not sent.")
    else:
        print(f"\n📤 Email sent! (after {state.get('revision_count', 0)} revision(s))")
    return state


def review_router(state: EmailState) -> str:
    """Route based on human review decision."""
    if state.get("approved", False):
        return "execute"
    return "revise"

In [ ]:
# Build the approval gate workflow
workflow = StateGraph(EmailState)

workflow.add_node("draft", draft_email)
workflow.add_node("review", human_review)
workflow.add_node("execute", execute_send)

workflow.set_entry_point("draft")
workflow.add_edge("draft", "review")
workflow.add_conditional_edges(
    "review",
    review_router,
    {"execute": "execute", "revise": "draft"},
)
workflow.add_edge("execute", END)

email_app = workflow.compile()
print("Email workflow compiled!")

In [ ]:
# Run it! You'll be prompted to approve/revise the draft.
result = email_app.invoke({
    "request": "Invite the AI Builders Lab participants to our March 18 session. "
               "Mention it's a hands-on collaborative session using Jupyter notebooks "
               "with pre-installed AI tools. Keep it short and enthusiastic.",
    "messages": [],
    "draft": "",
    "feedback": "",
    "approved": False,
    "revision_count": 0,
})

## Pattern 2: Confidence-Based Autonomy

Instead of always asking for approval, the agent self-assesses its confidence and only asks for review when uncertain.

```
[Classify] → [High confidence] → [Auto-execute] → END
                ↓
           [Low confidence] → [Human Review] → [Execute] → END
```

In [ ]:
import json


class TaskState(TypedDict):
    """State for confidence-based routing."""
    task: str
    messages: Annotated[list, add_messages]
    classification: str
    confidence: float
    response: str
    autonomy_level: str  # "auto" or "review"


def classify_task(state: TaskState) -> dict:
    """Classify the task and assess confidence."""
    prompt = (
        f"Classify this task and rate your confidence in handling it.\n\n"
        f"Task: {state['task']}\n\n"
        f"Respond in JSON format:\n"
        f'{{"classification": "routine|complex|sensitive", '
        f'"confidence": 0.0-1.0, '
        f'"reasoning": "brief explanation"}}'
    )

    response = model.invoke([HumanMessage(content=prompt)])

    try:
        # Extract JSON from response
        text = response.content
        # Handle markdown code blocks
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        parsed = json.loads(text.strip())
    except (json.JSONDecodeError, IndexError):
        parsed = {"classification": "complex", "confidence": 0.5, "reasoning": "Could not parse"}

    classification = parsed.get("classification", "complex")
    confidence = float(parsed.get("confidence", 0.5))

    # Determine autonomy level based on confidence + classification
    if classification == "routine" and confidence >= 0.8:
        autonomy = "auto"
    elif classification == "sensitive":
        autonomy = "review"  # Always review sensitive tasks
    elif confidence >= 0.9:
        autonomy = "auto"
    else:
        autonomy = "review"

    print(f"📊 Classification: {classification} | Confidence: {confidence:.0%} | → {autonomy}")
    print(f"   Reasoning: {parsed.get('reasoning', 'N/A')}")

    return {
        "classification": classification,
        "confidence": confidence,
        "autonomy_level": autonomy,
        "messages": [response],
    }


def generate_response(state: TaskState) -> dict:
    """Generate a response to the task."""
    prompt = f"Handle this task concisely:\n\n{state['task']}"
    response = model.invoke([HumanMessage(content=prompt)])

    if state["autonomy_level"] == "auto":
        print(f"\n🤖 Auto-executing (confidence was high):")
    else:
        print(f"\n📋 Proposed response (awaiting review):")

    print(response.content[:300] + ("..." if len(response.content) > 300 else ""))

    return {"response": response.content, "messages": [response]}


def human_checkpoint(state: TaskState) -> dict:
    """Human reviews the proposed response."""
    print("\n🧑 HUMAN CHECKPOINT — This task was flagged for review.")
    choice = input("[a]pprove / [r]eject: ").strip().lower()

    if choice == "a":
        print("✅ Approved")
    else:
        print("❌ Rejected — response discarded")
        return {"response": "[REJECTED BY HUMAN]"}

    return state


def autonomy_router(state: TaskState) -> str:
    """Route based on autonomy level."""
    return state.get("autonomy_level", "review")

In [ ]:
# Build the confidence-based workflow
workflow2 = StateGraph(TaskState)

workflow2.add_node("classify", classify_task)
workflow2.add_node("generate", generate_response)
workflow2.add_node("checkpoint", human_checkpoint)

workflow2.set_entry_point("classify")
workflow2.add_edge("classify", "generate")
workflow2.add_conditional_edges(
    "generate",
    autonomy_router,
    {"auto": END, "review": "checkpoint"},
)
workflow2.add_edge("checkpoint", END)

autonomy_app = workflow2.compile()
print("Confidence-based autonomy workflow compiled!")

In [ ]:
# Test with a ROUTINE task (should auto-execute)
print("=" * 50)
print("Test 1: Routine task")
print("=" * 50)
autonomy_app.invoke({
    "task": "What is 2 + 2?",
    "messages": [], "classification": "", "confidence": 0.0,
    "response": "", "autonomy_level": "",
})

In [ ]:
# Test with a SENSITIVE task (should require review)
print("=" * 50)
print("Test 2: Sensitive task")
print("=" * 50)
autonomy_app.invoke({
    "task": "Draft a public statement about our company's data breach incident",
    "messages": [], "classification": "", "confidence": 0.0,
    "response": "", "autonomy_level": "",
})

## 💡 Key Takeaways

1. **Start supervised, earn autonomy** — Begin with human approval for everything, then relax controls as trust builds
2. **Classify before acting** — Let the agent assess task complexity/sensitivity to determine the right autonomy level
3. **Always keep an escape hatch** — Even autonomous agents should have human override capability
4. **Log everything** — Every decision, every approval/rejection, for audit and improvement

## Design Principles for Production

- **Sensitive actions always require review** (data deletion, public communications, financial transactions)
- **Confidence thresholds are tunable** — start conservative (0.95), lower as the system proves reliable
- **Human feedback improves the system** — track rejection reasons to improve classification
- **Autonomy is per-task, not per-agent** — the same agent can be autonomous for routine tasks and supervised for sensitive ones

---

<div style="background-color:#1dc28b; color:white; padding:14px; border-radius:5px; margin-bottom:10px;">
<strong>🎉 Congratulations!</strong> You've completed the GovSpace Agentic Gym starter notebooks. You now have the building blocks for:
<ul style="margin-top:8px; margin-bottom:0;">
<li>Tool-calling agents (01)</li>
<li>Stateful multi-step workflows (02)</li>
<li>Human-in-the-loop graduated autonomy (03)</li>
</ul>
</div>

**What to build next:** Try combining these patterns — a LangGraph workflow with tools AND human checkpoints at critical decision points.

Explore more at [GovSpace Connect](https://app.govspace.io/connect) and [GovSpace Discover](https://app.govspace.io/discover/library?view=network).

---## 💡 You have reached the end of this super-notebook.Return to your **Module M2.10** in the GovSpace Academy to:- Post your Discussion Question response in the GovSpace Connect thread- Complete the Module Quiz- Reflect on which workflow in your own agency this pattern could apply to